# Create a Simple Reflex-Based Lunar Lander Agent

In this example, we will use Gymnasium, an environment to train agents via reinforcement learning (RL). We will not use RL here but just use the environment with a custom simple reflex-based agent.

## Install Gymnasium

The documentation for Gymnasium is available at https://gymnasium.farama.org/

Steps:
1. Create a new folder and open it with VS Code and install all needed Python Extensions in VS Code.
2. Create a new virtual environment (CTRL-Shift P Python Create Environment...)
3. I needed to install swig and the Python C++ headers on WSL2 via the terminal
    * `sudo apt install swig`
    * `sudo apt-get install python3-dev`
4. Install python libraries for gymnasium with the needed extras

In [10]:
# Google Colab or if you are missing a package: Uncomment the %pip command below, run this block, and restart the session.
# %pip install -q "swig>=4,<5" "gymnasium[box2d,classic-control]>=1.0,<2"

Additional installs for screen capturing so environment visualizations can be converted to embedded videos.
For recording videos, I had to install
    
* `sudo apt-get install xvfb ffmpeg`

Additional python packages for screen capturing.

In [11]:
# Google Colab or if you are missing a package: Uncomment the %pip command below, run this block, and restart the session.
# %pip install -q "pyvirtualdisplay>=3,<4" "gymnasium[other]>=1.0,<2"

## The Lunar Lander Environment

![Luna Lander image](https://gymnasium.farama.org/_images/lunar_lander.gif)

The documentation of the environment is available at: https://gymnasium.farama.org/environments/box2d/lunar_lander/

* Performance Measure: A reward of -100 or +100 points for crashing or landing safely respectively. We do not use
  intermediate rewards here.

* Environment: This environment is a classic rocket trajectory optimization problem. A ship needs to land safely. The space is **continuous** with
  x and y coordinates in the range [-2.5, 2.5]. The landing pad is at coordinate (0,0).

* Actuators:  According to Pontryagin’s
  maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off. There are four discrete actions available:

    - 0: do nothing
    - 1: fire left orientation engine
    - 2: fire main engine
    - 3: fire right orientation engine

* Sensors: Each observation is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

The different ranges/settings of the environment can be queried.

In [12]:
import gymnasium as gym
import numpy as np
np.set_printoptions(precision=2)

In [13]:
def query_environment(name):
    env = gym.make(name)
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {env.spec.max_episode_steps}")
    print(f"Nondeterministic: {env.spec.nondeterministic}")
   # print(f"Reward Range: {env.reward_range}")
    print(f"Reward Threshold: {env.spec.reward_threshold}")
    env.close()

query_environment("LunarLander-v3")

Action Space: Discrete(4)
Observation Space: Box([ -2.5   -2.5  -10.   -10.    -6.28 -10.    -0.    -0.  ], [ 2.5   2.5  10.   10.    6.28 10.    1.    1.  ], (8,), float32)
Max Episode Steps: 1000
Nondeterministic: False
Reward Threshold: 200



Gymnasium environments are implemented as classes with a `make` method to create the environment, a `reset` method, and a `step` method to execute an action.
To use it with an agent function that expects percepts and returns an action, we need write glue code that connects the environment with the agent function.

In [14]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    # run one episode
    for _ in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        if verbose:
            print (f"Obs: {observation} -> Action: {action}")

        # step: execute an action in the environment
        observation, reward, terminated, truncated, info = env.step(action)

        # render the environment
        if render:
            env.render()

        if terminated:
            if verbose:
                print(f"Final Reward: {reward}")
            break

    return reward

## Example: A Random Agent

We randomly return one of the actions. The environment accepts the integers 0-3.


In [15]:
def random_agent_function(observation):
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return np.random.choice([0, 1, 2, 3], p=[0.25, 0.25, 0.25, 0.25])

Run an episode.

In [16]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")

run_episode(random_agent_function, env)

env.close()

Obs: [ 0.    1.4   0.2  -0.52 -0.   -0.05  0.    0.  ] -> Action: 0
Obs: [ 0.    1.39  0.2  -0.55 -0.   -0.04  0.    0.  ] -> Action: 1
Obs: [ 5.84e-03  1.37e+00  1.89e-01 -5.73e-01 -4.51e-03  2.26e-04  0.00e+00
  0.00e+00] -> Action: 0
Obs: [ 7.73e-03  1.36e+00  1.89e-01 -6.00e-01 -4.50e-03  2.05e-04  0.00e+00
  0.00e+00] -> Action: 0
Obs: [ 9.62e-03  1.35e+00  1.89e-01 -6.27e-01 -4.49e-03  2.15e-04  0.00e+00
  0.00e+00] -> Action: 3
Obs: [ 0.01  1.33  0.2  -0.65 -0.01 -0.04  0.    0.  ] -> Action: 3
Obs: [ 0.01  1.32  0.21 -0.68 -0.01 -0.07  0.    0.  ] -> Action: 0
Obs: [ 0.02  1.3   0.21 -0.71 -0.01 -0.07  0.    0.  ] -> Action: 0
Obs: [ 0.02  1.28  0.21 -0.73 -0.02 -0.07  0.    0.  ] -> Action: 3
Obs: [ 0.02  1.27  0.22 -0.76 -0.02 -0.11  0.    0.  ] -> Action: 3
Obs: [ 0.02  1.25  0.23 -0.79 -0.03 -0.15  0.    0.  ] -> Action: 1
Obs: [ 0.02  1.23  0.22 -0.81 -0.04 -0.11  0.    0.  ] -> Action: 2
Obs: [ 0.03  1.21  0.2  -0.8  -0.04 -0.13  0.    0.  ] -> Action: 1
Obs: [ 0.03  1.19

Gymnasium displays environments using `render()` method on the local display. Headless installations like Google Colab do not have a display, but the output can be captured using a virtual display as a video and then add the video to the notebook.

We need to start a virtual display.

Now we can use wrappers for the environment to record the display. The functions are provided in
[display_record.py].



In [17]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py",
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Artificial_Intelligence/refs/heads/master/Agents/")

In [18]:
import gymnasium_display_recorder as dr

env = dr.gym_make('LunarLander-v3', 'LL1', render_fps=30)
run_episode(random_agent_function, env, verbose=False)
env.close()

dr.show('LL1')

Running on Windows, skipping pyvirtualdisplay (Xvfb not supported).


c:\Users\ACER\Documents\CODING\TTNTNC\Lab01\venv\lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at c:\Users\ACER\Documents\CODING\TTNTNC\Lab01\lab01 - Agent\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


## A Simple Reflex-Based Agent

To make the code easier to read, we use enumerations for actions (integers) and observations (index in the observation vector).

In [19]:
from enum import Enum

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7


Define a simple agent that uses the main thruster to reduce the falling speed if it gets too fast.

In [20]:
def rocket_agent_function(observation):
    """A simple agent function."""

    # run the main thruster, if the lander is falling too fast
    if observation[Obs.VY.value] < -.4:
        return Act.MAIN.value

    return Act.NO_OP.value

In [21]:
env = dr.gym_make('LunarLander-v3', 'LL2', render_fps=30)
run_episode(rocket_agent_function, env, verbose = False)
env.close()

dr.show('LL2')

## Evaluating the Agent

Run the agent on 100 problems and report the average reward.

In [22]:
def run_episodes(agent_function, env, n=1000):
    """Run multiple episodes with the given agent and return the rewards for each episode."""
    return [run_episode(agent_function, env, verbose=False, render=False) for _ in range(n)]

Run experiments.

In [23]:
env = gym.make("LunarLander-v3", render_mode=None)

rewards = run_episodes(rocket_agent_function, env)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

This is not great performance!


## Implement A Better Reflex-Based Agent

Build a better that uses its right and left thrusters to land the craft (more) safely. Test your agent function using 100 problems.

In [24]:
def better_agent_function(observation):
    x, y, vx, vy, angle, angular_velocity, leg_l, leg_r = observation

    # Góc nghiêng mục tiêu: nghiêng theo hướng cần bay để về gần x=0
    target_angle = np.clip(x * 0.5 + vx * 1.0, -0.4, 0.4)
    angle_todo = (target_angle - angle) * 0.5 - angular_velocity * 1.0

    # Độ cao mục tiêu: hạ xuống mặt đất
    target_hover = np.abs(x) * 0.55
    hover_todo = (target_hover - y) * 0.5 - vy * 0.5

    # Khi 1 chân đã chạm đất: không cắt máy đột ngột, chỉ tập trung hãm tốc độ rơi
    # để tránh tàu bị lật do chân còn lại vẫn lơ lửng
    if leg_l or leg_r:
        angle_todo = 0
        hover_todo = -vy * 0.5

    # Ưu tiên xử lý sai số nào lớn hơn: nếu lệch góc nhiều hơn lệch độ cao
    # thì chỉnh góc trước (vì lực máy chính đẩy theo trục thân tàu, nghiêng nhiều sẽ kém hiệu quả)
    if hover_todo > abs(angle_todo) and hover_todo > 0.05:
        return Act.MAIN.value
    elif angle_todo < -0.05:
        return Act.RIGHT.value  # phun động cơ phải -> xoay trái
    elif angle_todo > 0.05:
        return Act.LEFT.value   # phun động cơ trái -> xoay phải

    return Act.NO_OP.value

# Ghi hình 1 tập minh hoạ
env_vis = dr.gym_make('LunarLander-v3', 'LL3', render_fps=30)
run_episode(better_agent_function, env_vis, verbose=False, render=True)
env_vis.close()

dr.show('LL3')

Better Agent sử dụng các luật điều khiển dựa trên trạng thái của tàu:

* **`target_angle`**: xác định góc nghiêng mục tiêu dựa trên vị trí `x` và vận tốc ngang `vx`, giúp tàu quay về gần vị trí trung tâm.
* **`angle_todo`**: tính mức độ cần điều chỉnh góc, từ đó chọn **động cơ trái/phải** để giữ tàu cân bằng.
* **`target_hover` và `hover_todo`**: xác định độ cao mục tiêu và mức độ cần điều chỉnh độ cao, dùng **động cơ chính** để kiểm soát tốc độ rơi.
* **`leg_l`, `leg_r`**: khi một chân đã chạm đất, agent ưu tiên giảm vận tốc rơi để hạn chế tàu bị lật.
* Cuối cùng, agent chọn một trong bốn hành động: **MAIN, LEFT, RIGHT hoặc NO_OP** tùy theo sai lệch hiện tại.


## Đánh giá 3 agent trên 100 episode

In [26]:
def evaluate_agent(agent_function, name, n=100):
    """Chạy agent trên n episode và in báo cáo đầy đủ."""
    env = gym.make("LunarLander-v3", render_mode=None)
    rewards = run_episodes(agent_function, env, n=n)
    env.close()

    mean_r = np.mean(rewards)
    std_r = np.std(rewards)
    success_rate = np.sum(np.array(rewards) == 100) / n

    return {"name": name, "rewards": rewards, "mean": mean_r,
            "std": std_r, "success_rate": success_rate}


# Chạy cả 3 agent trên đúng 100 episode để so sánh công bằng
results_random = evaluate_agent(random_agent_function, "Random Agent")
results_rocket  = evaluate_agent(rocket_agent_function,  "Simple Reflex Agent (1 luật)")
results_best    = evaluate_agent(better_agent_function,    "Better Reflex Agent (cải tiến)")

# Bảng so sánh tổng hợp
print(f"{'Agent':<30}{'Mean Reward':>15}{'Std':>10}{'Success Rate':>15}")
for r in [results_random, results_rocket, results_best]:
    print(f"{r['name']:<30}{r['mean']:>15.2f}{r['std']:>10.2f}{r['success_rate']*100:>14.1f}%")

Agent                             Mean Reward       Std   Success Rate
Random Agent                          -100.00      0.00           0.0%
Simple Reflex Agent (1 luật)           -98.00     19.90           1.0%
Better Reflex Agent (cải tiến)          94.00     34.12          97.0%


NHẬN XÉT ĐỘ ỔN ĐỊNH:

- Random Agent: Kết quả không có biến động vì agent luôn thất bại và nhận reward gần như giống nhau trong các lần chạy. Vì vậy, Std = 0 không đồng nghĩa với hiệu quả ổn định.

- Simple Reflex Agent: So với Random Agent, reward trung bình có cải thiện nhẹ và đã xuất hiện một số lần hạ cánh thành công. Tuy nhiên, Std khá lớn so với Mean, cho thấy kết quả giữa các lần chạy còn biến động nhiều.

- Better Reflex Agent: Agent đạt reward trung bình và tỷ lệ thành công cao hơn rõ rệt. Tuy nhiên, Std = 34.12 khá lớn, nghĩa là reward giữa các episode có sự chênh lệch đáng kể. Cụ thể, có những episode agent hạ cánh tốt và nhận reward cao, trong khi một số episode dù vẫn thành công nhưng quá trình hạ cánh kém tối ưu hơn nên reward thấp hơn. Vì vậy, tỷ lệ thành công cao nhưng reward chưa hoàn toàn ổn định giữa các episode.

**Nhìn chung, Better Agent cho kết quả thành công cao và reward trung bình tốt, nhưng vẫn còn sự biến động về reward giữa các episode.**

## Vì sao tác tử phản xạ khó đạt hiệu năng cao trong môi trường liên tục?

Tác tử phản xạ chỉ lựa chọn hành động dựa trên **trạng thái hiện tại** và các **luật cục bộ**, nên có một số hạn chế:

- **Chỉ xét trạng thái hiện tại:** Không ghi nhớ trạng thái trước đó nên khó dự đoán xu hướng chuyển động tiếp theo.
- **Luật đơn giản:** Một luật phù hợp ở trạng thái này có thể không phù hợp khi vị trí, vận tốc hoặc góc thay đổi liên tục.
- **Không có khả năng lập kế hoạch:** Agent chủ yếu phản ứng với sai lệch hiện tại thay vì tối ưu toàn bộ quá trình hạ cánh.
- **Khó xử lý nhiều yếu tố đồng thời:** Vị trí, vận tốc ngang/dọc và góc nghiêng đều thay đổi liên tục và ảnh hưởng lẫn nhau.

**Do đó, tác tử phản xạ có thể hoạt động trong các tình huống đơn giản nhưng khó đạt hiệu năng cao và ổn định trong môi trường liên tục như LunarLander.**